# BBL Match Win Probability Prediction Using Checkpoint-Based Features

## 1. Library Import and Environment Setup

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score, accuracy_score


## 2. Data Loading and Initial Preparation


In [8]:
BASE = Path.cwd()
CHK_PATH = BASE / "data" / "bbl_checkpoint_wide_enhanced.csv"   # update path if needed

df = pd.read_csv(CHK_PATH)
df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

# remove any over20 columns to avoid terminal-state leakage
df = df.drop(columns=[c for c in df.columns if "over20" in c], errors="ignore")

# stable row id
df["match_idx"] = np.arange(len(df))

# target
target_col = "home_win"


## 3. Pre-Match Team Form Feature Engineering

In [11]:
matches = df[["match_idx", "date", "team1", "team2", "winner", "home_team"]].copy()

team1_hist = matches[["match_idx", "date", "team1", "winner"]].rename(columns={"team1": "team"})
team2_hist = matches[["match_idx", "date", "team2", "winner"]].rename(columns={"team2": "team"})

teams_df = pd.concat([team1_hist, team2_hist], ignore_index=True)
teams_df["win"] = (teams_df["team"] == teams_df["winner"]).astype(int)
teams_df = teams_df.sort_values(["team", "date", "match_idx"]).reset_index(drop=True)

# rolling win rate from last 5 prior matches only
teams_df["win_rate"] = (
    teams_df.groupby("team")["win"]
    .transform(lambda x: x.shift().rolling(5, min_periods=1).mean())
    .fillna(0.5)
)

team1_strength = (
    team1_hist[["match_idx", "team"]]
    .merge(
        teams_df[["match_idx", "team", "win_rate"]],
        on=["match_idx", "team"],
        how="left"
    )
    .rename(columns={"win_rate": "team1_strength"})
    [["match_idx", "team1_strength"]]
)

team2_strength = (
    team2_hist[["match_idx", "team"]]
    .merge(
        teams_df[["match_idx", "team", "win_rate"]],
        on=["match_idx", "team"],
        how="left"
    )
    .rename(columns={"win_rate": "team2_strength"})
    [["match_idx", "team2_strength"]]
)

df = df.merge(team1_strength, on="match_idx", how="left")
df = df.merge(team2_strength, on="match_idx", how="left")

## 4. Pre-Match Elo Rating Construction

In [14]:
teams = pd.unique(df[["team1", "team2"]].values.ravel())
elo = {team: 1500 for team in teams}

elo_team1_pre = []
elo_team2_pre = []

K = 20

for _, row in df.iterrows():
    t1 = row["team1"]
    t2 = row["team2"]

    r1 = elo[t1]
    r2 = elo[t2]

    # store PRE-match Elo
    elo_team1_pre.append(r1)
    elo_team2_pre.append(r2)

    # expected score
    e1 = 1 / (1 + 10 ** ((r2 - r1) / 400))
    s1 = 1 if row["winner"] == t1 else 0

    # update after match
    elo[t1] = r1 + K * (s1 - e1)
    elo[t2] = r2 + K * ((1 - s1) - (1 - e1))

df["elo_team1"] = elo_team1_pre
df["elo_team2"] = elo_team2_pre


## 5. Home–Away Feature Orientation

In [17]:
df["away_team"] = np.where(df["home_team"] == df["team1"], df["team2"], df["team1"])

df["home_strength"] = np.where(df["home_team"] == df["team1"], df["team1_strength"], df["team2_strength"])
df["away_strength"] = np.where(df["home_team"] == df["team1"], df["team2_strength"], df["team1_strength"])
df["strength_diff_home"] = df["home_strength"] - df["away_strength"]

df["home_elo"] = np.where(df["home_team"] == df["team1"], df["elo_team1"], df["elo_team2"])
df["away_elo"] = np.where(df["home_team"] == df["team1"], df["elo_team2"], df["elo_team1"])
df["elo_diff_home"] = df["home_elo"] - df["away_elo"]


## 6. Checkpoint Feature Engineering

In [20]:
for over in [6, 10, 15]:
    # home/away runs
    df[f"home_runs_over{over}"] = np.where(
        df[f"home_is_batting_inn1_over{over}"] == 1,
        df[f"cum_runs_inn1_over{over}"],
        df[f"cum_runs_inn2_over{over}"]
    )
    df[f"away_runs_over{over}"] = np.where(
        df[f"home_is_batting_inn1_over{over}"] == 1,
        df[f"cum_runs_inn2_over{over}"],
        df[f"cum_runs_inn1_over{over}"]
    )

    # home/away wickets in hand
    df[f"home_wkts_in_hand_over{over}"] = np.where(
        df[f"home_is_batting_inn1_over{over}"] == 1,
        df[f"wkts_in_hand_inn1_over{over}"],
        df[f"wkts_in_hand_inn2_over{over}"]
    )
    df[f"away_wkts_in_hand_over{over}"] = np.where(
        df[f"home_is_batting_inn1_over{over}"] == 1,
        df[f"wkts_in_hand_inn2_over{over}"],
        df[f"wkts_in_hand_inn1_over{over}"]
    )

    # home/away run rate
    df[f"home_run_rate_over{over}"] = np.where(
        df[f"home_is_batting_inn1_over{over}"] == 1,
        df[f"run_rate_inn1_over{over}"],
        df[f"run_rate_inn2_over{over}"]
    )
    df[f"away_run_rate_over{over}"] = np.where(
        df[f"home_is_batting_inn1_over{over}"] == 1,
        df[f"run_rate_inn2_over{over}"],
        df[f"run_rate_inn1_over{over}"]
    )

    # difference features in home perspective
    df[f"score_diff_home_over{over}"] = df[f"home_runs_over{over}"] - df[f"away_runs_over{over}"]
    df[f"wkts_diff_home_over{over}"] = df[f"home_wkts_in_hand_over{over}"] - df[f"away_wkts_in_hand_over{over}"]
    df[f"rr_diff_home_over{over}"] = df[f"home_run_rate_over{over}"] - df[f"away_run_rate_over{over}"]

    # chase pressure in home perspective
    df[f"home_pressure_over{over}"] = np.where(
        df[f"home_is_batting_inn2_over{over}"] == 1,
        df[f"req_run_rate_inn2_over{over}"] / (df[f"run_rate_inn2_over{over}"] + 0.01),
        np.nan
    )

    df[f"away_pressure_over{over}"] = np.where(
        df[f"home_is_batting_inn2_over{over}"] == 0,
        df[f"req_run_rate_inn2_over{over}"] / (df[f"run_rate_inn2_over{over}"] + 0.01),
        np.nan
    )

    df[f"home_pressure_over{over}"] = df[f"home_pressure_over{over}"].fillna(1.0)
    df[f"away_pressure_over{over}"] = df[f"away_pressure_over{over}"].fillna(1.0)

    df[f"pressure_diff_home_over{over}"] = df[f"home_pressure_over{over}"] - df[f"away_pressure_over{over}"]



## 7. Train–Test Split


In [23]:
train = df[df["date"] < "2020-01-01"].copy()
test = df[df["date"] >= "2020-01-01"].copy()

y_train = train[target_col]
y_test = test[target_col]

## 8. Model Evaluation 

In [26]:
def evaluate(y_true, y_pred, label):
    print(f"\n--- {label} ---")
    print(f"Log Loss : {log_loss(y_true, y_pred):.4f}")
    print(f"Brier    : {brier_score_loss(y_true, y_pred):.4f}")
    print(f"AUC      : {roc_auc_score(y_true, y_pred):.4f}")
    print(f"Accuracy : {accuracy_score(y_true, (y_pred > 0.5)):.4f}")


## 9. Logistic Regression Modelling

In [31]:
def run_lr_model(over):
    print(f"\n==============================")
    print(f"LOGISTIC REGRESSION FOR OVER {over}")
    print(f"==============================")

    feature_candidates = [
        f"score_diff_home_over{over}",
        f"wkts_diff_home_over{over}",
        f"rr_diff_home_over{over}",
        f"pressure_diff_home_over{over}",
        "strength_diff_home",
        "elo_diff_home",
        f"home_is_batting_inn1_over{over}",
        f"home_is_batting_inn2_over{over}",
    ]

    features = [c for c in feature_candidates if c in train.columns and c in test.columns]

    X_train = train[features].copy()
    X_test = test[features].copy()

    lr_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=1000, C=0.5))
    ])

    lr_model.fit(X_train, y_train)
    y_pred = lr_model.predict_proba(X_test)[:, 1]

    evaluate(y_test, y_pred, f"LR Over {over}")

    coef = lr_model.named_steps["lr"].coef_[0]
    coef_df = pd.DataFrame({
        "feature": features,
        "coefficient": coef
    }).sort_values("coefficient", ascending=False)

    print("\nCoefficients:")
    print(coef_df.to_string(index=False))

    return lr_model, coef_df


## 10. Checkpoint Model Training and Testing

In [34]:
lr_6, coef_6 = run_lr_model(6)
lr_10, coef_10 = run_lr_model(10)
lr_15, coef_15 = run_lr_model(15)


LOGISTIC REGRESSION FOR OVER 6

--- LR Over 6 ---
Log Loss : 0.5855
Brier    : 0.1901
AUC      : 0.7843
Accuracy : 0.7055

Coefficients:
                   feature  coefficient
      wkts_diff_home_over6     0.802448
home_is_batting_inn2_over6     0.459338
             elo_diff_home     0.214600
        strength_diff_home     0.080757
home_is_batting_inn1_over6    -0.062894
     score_diff_home_over6    -0.066988
        rr_diff_home_over6    -0.066988
  pressure_diff_home_over6    -1.384728

LOGISTIC REGRESSION FOR OVER 10

--- LR Over 10 ---
Log Loss : 0.5360
Brier    : 0.1790
AUC      : 0.8107
Accuracy : 0.7282

Coefficients:
                    feature  coefficient
      wkts_diff_home_over10     1.002651
home_is_batting_inn2_over10     0.574834
              elo_diff_home     0.299982
        rr_diff_home_over10     0.169339
     score_diff_home_over10     0.169339
         strength_diff_home     0.024664
home_is_batting_inn1_over10    -0.122142
  pressure_diff_home_over10    -1.

## 11. Tuned Random Forest Modelling

In [37]:
def run_rf_gridsearch(over):
    print(f"\n==============================")
    print(f"GRID SEARCH RF FOR OVER {over}")
    print(f"==============================")

    feature_candidates = [
        f"score_diff_home_over{over}",
        f"wkts_diff_home_over{over}",
        f"rr_diff_home_over{over}",
        f"pressure_diff_home_over{over}",
        "strength_diff_home",
        "elo_diff_home",
        f"home_is_batting_inn1_over{over}",
        f"home_is_batting_inn2_over{over}",
    ]

    features = [c for c in feature_candidates if c in train.columns and c in test.columns]

    X_train = train[features].copy()
    X_test = test[features].copy()

    pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("rf", RandomForestClassifier(random_state=42))
    ])

    param_grid = {
        "rf__n_estimators": [200, 400, 600],
        "rf__max_depth": [4, 6, 8, None],
        "rf__min_samples_leaf": [1, 3, 5, 10],
        "rf__max_features": ["sqrt", "log2", None]
    }

    grid = GridSearchCV(
        estimator=pipe,
        param_grid=param_grid,
        cv=5,
        scoring="roc_auc",
        n_jobs=-1,
        verbose=1
    )

    grid.fit(X_train, y_train)

    print("\nBest parameters:")
    print(grid.best_params_)
    print(f"Best CV AUC: {grid.best_score_:.4f}")

    best_model = grid.best_estimator_
    y_pred = best_model.predict_proba(X_test)[:, 1]

    evaluate(y_test, y_pred, f"Tuned RF Over {over}")

    importances = best_model.named_steps["rf"].feature_importances_
    imp_df = pd.DataFrame({
        "feature": features,
        "importance": importances
    }).sort_values("importance", ascending=False)

    print("\nFeature importances:")
    print(imp_df.to_string(index=False))

    return grid, best_model, imp_df


## 12. Hyperparameter Tuning with Grid Search

In [40]:
grid_6, tuned_rf_6, imp_6 = run_rf_gridsearch(6)
grid_10, tuned_rf_10, imp_10 = run_rf_gridsearch(10)
grid_15, tuned_rf_15, imp_15 = run_rf_gridsearch(15)


GRID SEARCH RF FOR OVER 6
Fitting 5 folds for each of 144 candidates, totalling 720 fits

Best parameters:
{'rf__max_depth': 4, 'rf__max_features': None, 'rf__min_samples_leaf': 1, 'rf__n_estimators': 200}
Best CV AUC: 0.8307

--- Tuned RF Over 6 ---
Log Loss : 0.6113
Brier    : 0.2074
AUC      : 0.7495
Accuracy : 0.6990

Feature importances:
                   feature  importance
  pressure_diff_home_over6    0.521590
      wkts_diff_home_over6    0.209199
             elo_diff_home    0.115957
     score_diff_home_over6    0.045506
        rr_diff_home_over6    0.040659
        strength_diff_home    0.034858
home_is_batting_inn2_over6    0.023378
home_is_batting_inn1_over6    0.008852

GRID SEARCH RF FOR OVER 10
Fitting 5 folds for each of 144 candidates, totalling 720 fits

Best parameters:
{'rf__max_depth': 4, 'rf__max_features': None, 'rf__min_samples_leaf': 10, 'rf__n_estimators': 600}
Best CV AUC: 0.8561

--- Tuned RF Over 10 ---
Log Loss : 0.5119
Brier    : 0.1712
AUC      : 0

## 13. Performance Comparison and Interpretation

The final comparison shows that both models performed strongly on the enhanced wide dataset, with predictive performance improving as the checkpoint moved later in the match. Logistic Regression provided a strong and interpretable baseline, achieving AUC values of 0.7843 at over 6, 0.8107 at over 10, and 0.8875 at over 15, with the best overall result at over 15 (accuracy 0.7735). The tuned Random Forest served as the non-linear benchmark and performed competitively, with AUC values of 0.7495 at over 6, 0.8251 at over 10, and 0.8714 at over 15.

Overall, Logistic Regression was the strongest model at the later checkpoint, while the tuned Random Forest was most competitive at over 10. This suggests that non-linear interactions are useful earlier and mid-innings, whereas by over 15 the match state becomes more linearly separable, favouring Logistic Regression. Across both models, the main predictive signals were wicket difference, score difference, pressure, and Elo-based team strength. These results indicate that the enhanced wide dataset successfully preserved enough cricket-specific context to recover strong checkpoint-based win probability performance while also addressing the row-independence issue raised for Logistic Regression.